In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial.distance import cosine, pdist
from typing import Dict, Tuple
from dataclasses import dataclass

In [34]:
# Setting paths
data_dir = Path("../Data/20260116 Comparison 3")
output_dir = Path("Comparison3_Figures")
output_dir.mkdir(exist_ok=True)

In [35]:
# Dextramers
markers = [
        "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
        "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
        "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
        "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
        "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
        "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    	"RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
        "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
        "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
        "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
        "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
        "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
        "RiO-Allo:H-2Kb-VSFTYRYL-pAbO"
        ]

# Peptides on dextramers
peptides = [
    "ATLVFHNL",
    "EEEPVKKI",
    "HIYEFPQL",
    "INFDFPKL",
    "RAYLFNSV",
    "RTYTYEKL",
    "SNYLFTKL",
    "SSYTFPKM",
    "SVYVYKVL",
    "VAFDFTKV",
    "VGPRYTNL",
    "VIVRFLTV",
    "VSFTYRYL"
]

ly49c_col = "Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"
ct_col = "TCRClonotype"
sample_col = "Sample_Origin"

min_size = 5
plots_per_page = 32
thresholds = np.linspace(start=1.0, stop=0.85, num=16)

In [36]:
def shannon_entropy(p):
    """
    Normalised Shannon entropy
    """
    X = np.asarray(p, dtype=float)

    # Case 1: single vector
    if X.ndim == 1:
        s = X.sum()
        if s <= 0:
            return 0.0
        P = X / s
        P = P[P > 0]
        if P.size == 0:
            return 0.0
        return float(-np.sum(P * np.log2(P)))

    # Case 2: matrix of vectors (rows)
    if X.ndim == 2:
        row_sums = X.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P = X / row_sums

        with np.errstate(divide="ignore", invalid="ignore"):
            logP = np.where(P > 0, np.log2(P), 0.0)
        return -(P * logP).sum(axis=1)

    raise ValueError(f"Expected 1D or 2D input, got shape {X.shape}")


In [37]:
def top_peptide_dominance(x):
    """
    Top-peptide dominance: D = max(x) / sum(x)
    """
    X = np.asarray(x, dtype=float)

    if X.ndim == 1:
        total = X.sum()
        if total <= 0:
            return 0.0
        return float(X.max() / total)

    if X.ndim == 2:
        totals = X.sum(axis=1)
        totals[totals == 0] = 1.0
        return X.max(axis=1) / totals

    raise ValueError(f"Expected 1D or 2D input, got shape {X.shape}")


In [38]:
def pattern_distance(p1, p2, metric):
    """
    Distance between two normalised dextramer patterns.

    Cosine distance: measures change in pattern direction, ignores magnitude.
    L1 distance: measures total redistribution of binding.
    """
    if metric == 'cosine':
        if np.allclose(p1, 0) or np.allclose(p2, 0):
            return 1.0
        return float(cosine(p1, p2))
    elif metric == 'l1':
        return float(np.sum(np.abs(p1 - p2)))
    else:
        raise ValueError(f"Unknown metric: {metric}")

In [39]:
def within_clonotype_cosine_similarity(dex_matrix):
    """
    Mean pairwise cosine similarity between cells in a clonotype.

    Returns: (mean_similarity, std_similarity)
    """
    X = np.asarray(dex_matrix, dtype=float)
    if X.shape[0] < 2:
        return np.nan, np.nan

    row_sums = X.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    P = X / row_sums

    cos_dists = pdist(P, metric='cosine')
    cos_sims = 1 - cos_dists
    return float(np.mean(cos_sims)), float(np.std(cos_sims))


In [40]:
@dataclass
class ClonotypeSummary:
    """Container for clonotype-level metrics."""
    mean_pattern: np.ndarray
    sem: np.ndarray
    n_cells: int
    mean_entropy: float
    mean_dominance: float
    cosine_similarity: float
    cosine_similarity_std: float
    top_peptide_idx: int

def compute_clonotype_summaries(df, ct_col, markers, min_size=5):
    """
    Compute clonotype-level summaries for clonotypes with >= min_size cells.

    Returns:
      dict: clonotype -> ClonotypeSummary
    """
    ct_counts = df[ct_col].value_counts()
    valid_clonotypes = ct_counts[ct_counts >= min_size].index
    df_filtered = df[df[ct_col].isin(valid_clonotypes)]

    if len(df_filtered) == 0:
        return {}

    summaries = {}

    for ct, group in df_filtered.groupby(ct_col):
        dex_matrix = group[markers].to_numpy(dtype=float, copy=False)
        n_cells = dex_matrix.shape[0]

        # Normalise each cell vector
        row_sums = dex_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        dex_normed = dex_matrix / row_sums

        # Vectorised cell metrics
        entropies = shannon_entropy(dex_normed)            # returns (n_cells,)
        dominances = top_peptide_dominance(dex_matrix)     # returns (n_cells,)

        # Clonotype mean pattern + SEM
        mean_pattern = dex_normed.mean(axis=0)
        if n_cells > 1:
            sem = dex_normed.std(axis=0, ddof=1) / np.sqrt(n_cells)
        else:
            sem = np.zeros(dex_normed.shape[1], dtype=float)

        cos_sim, cos_sim_std = within_clonotype_cosine_similarity(dex_matrix)

        summaries[ct] = ClonotypeSummary(
            mean_pattern=mean_pattern,
            sem=sem,
            n_cells=int(n_cells),
            mean_entropy=float(np.mean(entropies)),
            mean_dominance=float(np.mean(dominances)),
            cosine_similarity=cos_sim,
            cosine_similarity_std=cos_sim_std,
            top_peptide_idx=int(np.argmax(mean_pattern))
        )

    return summaries


def apply_persample_ly49c_filter(df, percentile, ly49c_col, sample_col):
    """
    Per-sample Ly49C thresholding.

    For each sample s:
      T_s(q) = quantile_q(Ly49C | sample=s)

    Keep cells where:
      Ly49C <= T_{sample}(q)
    """
    if percentile >= 1.0:
        return df.copy()

    thresholds = df.groupby(sample_col)[ly49c_col].transform(
        lambda x: x.quantile(percentile)
    )
    return df[df[ly49c_col] <= thresholds].copy()


def compute_pattern_distances(baseline, filtered, metric='cosine'):
    """
    For clonotypes present in both baseline and filtered:
      distance = pattern_distance(mean_pattern_baseline, mean_pattern_filtered)

    Returns:
      DataFrame with per-clonotype distances and supporting metrics.
    """
    common = set(baseline.keys()) & set(filtered.keys())

    rows = []
    for ct in common:
        b = baseline[ct]
        f = filtered[ct]
        dist = pattern_distance(b.mean_pattern, f.mean_pattern, metric)

        rows.append({
            'clonotype': ct,
            'distance': dist,
            'n_cells_baseline': b.n_cells,
            'n_cells_filtered': f.n_cells,
            'entropy_baseline': b.mean_entropy,
            'entropy_filtered': f.mean_entropy,
            'similarity_baseline': b.cosine_similarity,
            'similarity_filtered': f.cosine_similarity
        })

    return pd.DataFrame(rows)

In [41]:
df = pd.read_csv(data_dir / "20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv", index_col=0)
df = df.sort_values(by=["CTCount", ly49c_col], ascending=[False, False])

print(f"Total cells: {len(df)}")
print(f"Unique clonotypes: {df[ct_col].nunique()}")
print(f"Samples: {df[sample_col].unique()}")

baseline_summaries = compute_clonotype_summaries(df, ct_col, markers, min_size)
print(f"Baseline: {len(baseline_summaries)} clonotypes with ≥{min_size} cells")

Total cells: 18924
Unique clonotypes: 5800
Samples: ['BL6-B10BR_HTxC' 'BL6-B10BR_HTxB' 'BL6-B10BR_HTxA']
Baseline: 470 clonotypes with ≥5 cells


In [42]:
sweep_results = []

for pct in thresholds:
    filtered_df = apply_persample_ly49c_filter(df, pct, ly49c_col, sample_col)
    filtered_summaries = compute_clonotype_summaries(filtered_df, ct_col, markers, min_size)

    dist_df = compute_pattern_distances(baseline_summaries, filtered_summaries, 'cosine')
    dist_df_l1 = compute_pattern_distances(baseline_summaries, filtered_summaries, 'l1')

    if len(dist_df) > 0:
        result = {
            'percentile': pct,
            'n_cells': len(filtered_df),
            'pct_cells_retained': 100 * len(filtered_df) / len(df),
            'n_clonotypes': len(filtered_summaries),
            'median_cosine_dist': dist_df['distance'].median(),
            'q25_cosine_dist': dist_df['distance'].quantile(0.25),
            'q75_cosine_dist': dist_df['distance'].quantile(0.75),
            'median_l1_dist': dist_df_l1['distance'].median(),
            'mean_entropy_change': (dist_df['entropy_filtered'] - dist_df['entropy_baseline']).mean(),
            'mean_similarity_change': (dist_df['similarity_filtered'] - dist_df['similarity_baseline']).mean(),
        }
    else:
        result = {
            'percentile': pct,
            'n_cells': np.nan,
            'pct_cells_retained': np.nan,
            'n_clonotypes': np.nan,
            'median_cosine_dist': np.nan,
            'q25_cosine_dist': np.nan,
            'q75_cosine_dist': np.nan,
            'median_l1_dist': np.nan,
            'mean_entropy_change': np.nan,
            'mean_similarity_change': np.nan,
        }

    sweep_results.append(result)

    if pct in thresholds:
        print(f"  q={pct:.2f}: {result['n_cells']:.0f} cells, "
              f"{result['n_clonotypes']:.0f} clonotypes, "
              f"Δ(q)={result['median_cosine_dist']:.4f}")

sweep_df = pd.DataFrame(sweep_results)

  q=1.00: 18924 cells, 470 clonotypes, Δ(q)=0.0000
  q=0.99: 18757 cells, 468 clonotypes, Δ(q)=0.0000
  q=0.98: 18753 cells, 468 clonotypes, Δ(q)=0.0000
  q=0.97: 18479 cells, 465 clonotypes, Δ(q)=0.0000
  q=0.96: 18468 cells, 465 clonotypes, Δ(q)=0.0000
  q=0.95: 18468 cells, 465 clonotypes, Δ(q)=0.0000
  q=0.94: 18468 cells, 465 clonotypes, Δ(q)=0.0000
  q=0.93: 17984 cells, 459 clonotypes, Δ(q)=0.0000
  q=0.92: 17984 cells, 459 clonotypes, Δ(q)=0.0000
  q=0.91: 17555 cells, 448 clonotypes, Δ(q)=0.0000
  q=0.90: 17521 cells, 447 clonotypes, Δ(q)=0.0000
  q=0.89: 17521 cells, 447 clonotypes, Δ(q)=0.0000
  q=0.88: 17521 cells, 447 clonotypes, Δ(q)=0.0000
  q=0.87: 17521 cells, 447 clonotypes, Δ(q)=0.0000
  q=0.86: 17521 cells, 447 clonotypes, Δ(q)=0.0000
  q=0.85: 17521 cells, 447 clonotypes, Δ(q)=0.0000


In [43]:
sweep_df['stability_gain'] = -sweep_df['median_cosine_dist'].diff()

max_gain = sweep_df['stability_gain'].max()
if not np.isnan(max_gain) and max_gain > 0:
    stable_mask = sweep_df['stability_gain'] < 0.1 * max_gain
    stable_indices = sweep_df[stable_mask].index
    if len(stable_indices) > 0:
        knee_idx = stable_indices[0]
        optimal_percentile = float(sweep_df.loc[knee_idx, 'percentile'])
        print(f"Knee detected at percentile: {optimal_percentile:.3f}")
    else:
        optimal_percentile = 0.95
        print(f"No clear knee; using default: {optimal_percentile}")
else:
    optimal_percentile = 0.95
    print(f"Insufficient variation; using default: {optimal_percentile}")

Insufficient variation; using default: 0.95


In [44]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(sweep_df['percentile'], sweep_df['median_cosine_dist'], 'o-', lw=2, ms=4)
ax.fill_between(sweep_df['percentile'],
                sweep_df['q25_cosine_dist'],
                sweep_df['q75_cosine_dist'], alpha=0.3)
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7, label=f'Knee: {optimal_percentile:.2f}')
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('Median Δ(q) [Cosine Distance]')
ax.set_title('Pattern Distance from Baseline')
ax.invert_xaxis()
ax.legend()

ax = axes[0, 1]
ax.plot(sweep_df['percentile'], sweep_df['median_l1_dist'], 'o-', lw=2, ms=4, color='orange')
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7)
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('Median Δ(q) [L1 Distance]')
ax.set_title('Pattern Distance - L1')
ax.invert_xaxis()

ax = axes[0, 2]
ax.plot(sweep_df['percentile'], sweep_df['stability_gain'], 'o-', lw=2, ms=4, color='green')
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7)
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('Stability Gain')
ax.set_title('Marginal Benefit of Filtering')
ax.invert_xaxis()

ax = axes[1, 0]
ax.plot(sweep_df['percentile'], sweep_df['mean_entropy_change'], 'o-', lw=2, ms=4, color='purple')
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7)
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('Mean Entropy Change')
ax.set_title('Specificity Change\n(negative = more specific)')
ax.invert_xaxis()

ax = axes[1, 1]
ax.plot(sweep_df['percentile'], sweep_df['mean_similarity_change'], 'o-', lw=2, ms=4, color='teal')
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7)
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('Mean Similarity Change')
ax.set_title('Within-Clonotype Coherence\n(positive = more coherent)')
ax.invert_xaxis()

ax = axes[1, 2]
ax.plot(sweep_df['percentile'], sweep_df['pct_cells_retained'], 'o-', lw=2, ms=4, label='% Cells')
ax.axvline(optimal_percentile, color='red', ls='--', alpha=0.7)
ax.set_xlabel('Ly49C Percentile Threshold')
ax.set_ylabel('% Retained')
ax.set_title('Data Retention')
ax.invert_xaxis()
ax.legend()

plt.tight_layout()
plt.savefig(output_dir / 'ly49c_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.savefig(output_dir / 'ly49c_threshold_sweep.pdf', bbox_inches='tight')
plt.close()
print(f"  Saved: {output_dir / 'ly49c_threshold_sweep.png'}")

  Saved: Comparison3_Figures/ly49c_threshold_sweep.png


In [45]:
CHOSEN_PERCENTILE = optimal_percentile

print(f"Threshold q = {CHOSEN_PERCENTILE:.3f}")

final_df = apply_persample_ly49c_filter(df, CHOSEN_PERCENTILE, ly49c_col, sample_col)
final_summaries = compute_clonotype_summaries(final_df, ct_col, markers, min_size)

print(f"Final: {len(final_df)} cells ({100*len(final_df)/len(df):.1f}% retained)")
print(f"Final: {len(final_summaries)} clonotypes")

final_df.to_csv(output_dir / f'filtered_ly49c_pct{int(CHOSEN_PERCENTILE*100)}.csv')

Threshold q = 0.950
Final: 18468 cells (97.6% retained)
Final: 465 clonotypes


In [46]:
def plot_clonotype_profiles(summaries, peptides, output_path, plots_per_page=32, title_suffix=""):
    """Generate multi-page PDF with SEM error bars for ALL clonotypes."""

    sorted_cts = sorted(summaries.keys(), key=lambda x: summaries[x].n_cells, reverse=True)
    xs = np.arange(len(peptides))

    n_pages = (len(sorted_cts) + plots_per_page - 1) // plots_per_page
    print(f"    Generating {n_pages} pages for {len(sorted_cts)} clonotypes...")

    with PdfPages(output_path) as pdf:
        for start in range(0, len(sorted_cts), plots_per_page):
            chunk = sorted_cts[start:start + plots_per_page]

            fig, axes = plt.subplots(8, 4, figsize=(18, 30))
            axes = axes.flatten()

            for ax_i, ct in enumerate(chunk):
                ax = axes[ax_i]
                s = summaries[ct]

                ax.bar(xs, s.mean_pattern, color='steelblue', edgecolor='none')
                ax.errorbar(xs, s.mean_pattern, yerr=s.sem,
                            fmt='none', capsize=2, lw=0.8, ecolor='black')

                ax.bar(s.top_peptide_idx, s.mean_pattern[s.top_peptide_idx],
                       color='darkred', edgecolor='none')

                ax.set_title(
                    f"{ct[:45]}{'...' if len(ct) > 45 else ''}\n"
                    f"n={s.n_cells}, H={s.mean_entropy:.2f}, D={s.mean_dominance:.2f}",
                    fontsize=6
                )
                ax.set_xticks(xs)
                ax.set_xticklabels(peptides, rotation=90, fontsize=5)
                ax.tick_params(axis='y', labelsize=5)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

            for j in range(len(chunk), plots_per_page):
                axes[j].axis('off')

            page_num = start // plots_per_page + 1
            fig.suptitle(f"Clonotype Dextramer Profiles {title_suffix} (Page {page_num}/{n_pages})",
                         fontsize=12, y=1.0)
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    print(f"  Saved: {output_path}")


plot_clonotype_profiles(
    baseline_summaries, peptides,
    output_dir / 'clonotypes_baseline_ALL.pdf',
    plots_per_page, "(Baseline)"
)

plot_clonotype_profiles(
    final_summaries, peptides,
    output_dir / f'clonotypes_filtered_pct{int(CHOSEN_PERCENTILE*100)}_ALL.pdf',
    plots_per_page, f"(Filtered: q={CHOSEN_PERCENTILE:.2f})"
)

    Generating 15 pages for 470 clonotypes...
  Saved: Comparison3_Figures/clonotypes_baseline_ALL.pdf
    Generating 15 pages for 465 clonotypes...
  Saved: Comparison3_Figures/clonotypes_filtered_pct95_ALL.pdf


In [47]:
df_by_ct = {ct: g for ct, g in df.groupby(ct_col)}

all_stats = []

for ct in baseline_summaries.keys():
    b = baseline_summaries[ct]

    ct_cells = df_by_ct[ct]
    ct_ly49c = ct_cells[ly49c_col].to_numpy(dtype=float, copy=False)

    n_ly49c_positive = int(np.sum(ct_ly49c > 0))
    pct_ly49c_positive = 100 * n_ly49c_positive / b.n_cells

    if ct in final_summaries:
        f = final_summaries[ct]
        n_cells_filtered = f.n_cells
        entropy_filtered = f.mean_entropy
        dominance_filtered = f.mean_dominance
        similarity_filtered = f.cosine_similarity
        pattern_dist = pattern_distance(b.mean_pattern, f.mean_pattern, 'cosine')
        pattern_dist_l1 = pattern_distance(b.mean_pattern, f.mean_pattern, 'l1')
    else:
        n_cells_filtered = 0
        entropy_filtered = np.nan
        dominance_filtered = np.nan
        similarity_filtered = np.nan
        pattern_dist = np.nan
        pattern_dist_l1 = np.nan

    top_idx = b.top_peptide_idx
    top_peptide = peptides[top_idx]

    row = {
        'clonotype': ct,
        'top_peptide': top_peptide,
        'top_peptide_idx': top_idx,

        'n_cells_baseline': b.n_cells,
        'n_cells_filtered': n_cells_filtered,
        'n_cells_removed': b.n_cells - n_cells_filtered,
        'pct_cells_retained': 100 * n_cells_filtered / b.n_cells if b.n_cells > 0 else 0,

        'n_ly49c_positive': n_ly49c_positive,
        'pct_ly49c_positive': pct_ly49c_positive,
        'ly49c_mean': float(np.mean(ct_ly49c)),
        'ly49c_median': float(np.median(ct_ly49c)),
        'ly49c_max': float(np.max(ct_ly49c)),
        'ly49c_sum': float(np.sum(ct_ly49c)),

        'entropy_baseline': b.mean_entropy,
        'entropy_filtered': entropy_filtered,
        'entropy_change': entropy_filtered - b.mean_entropy if not np.isnan(entropy_filtered) else np.nan,

        'dominance_baseline': b.mean_dominance,
        'dominance_filtered': dominance_filtered,
        'dominance_change': dominance_filtered - b.mean_dominance if not np.isnan(dominance_filtered) else np.nan,

        'cosine_similarity_baseline': b.cosine_similarity,
        'cosine_similarity_filtered': similarity_filtered,
        'similarity_change': similarity_filtered - b.cosine_similarity if not np.isnan(similarity_filtered) else np.nan,

        'pattern_distance_cosine': pattern_dist,
        'pattern_distance_l1': pattern_dist_l1,
    }

    for i, pep in enumerate(peptides):
        row[f'mean_{pep}_baseline'] = b.mean_pattern[i]
        row[f'sem_{pep}_baseline'] = b.sem[i]
        if ct in final_summaries:
            row[f'mean_{pep}_filtered'] = final_summaries[ct].mean_pattern[i]
            row[f'sem_{pep}_filtered'] = final_summaries[ct].sem[i]
        else:
            row[f'mean_{pep}_filtered'] = np.nan
            row[f'sem_{pep}_filtered'] = np.nan

    all_stats.append(row)

stats_df = pd.DataFrame(all_stats)
stats_df = stats_df.sort_values('n_cells_baseline', ascending=False).reset_index(drop=True)

stats_df.to_csv(output_dir / 'clonotype_summary_statistics_ALL.csv', index=False)
print(f"  Saved: {output_dir / 'clonotype_summary_statistics_ALL.csv'}")
print(f"  Contains {len(stats_df)} clonotypes with {len(stats_df.columns)} columns")

  Saved: Comparison3_Figures/clonotype_summary_statistics_ALL.csv
  Contains 470 clonotypes with 76 columns


In [49]:
print(f"\n  CLONOTYPE COUNTS:")
print(f"    Total clonotypes (≥{min_size} cells): {len(stats_df)}")
print(f"    Clonotypes retained after filtering: {(stats_df['n_cells_filtered'] >= min_size).sum()}")

print(f"\n  CELL COUNTS:")
print(f"    Total cells in clonotypes: {stats_df['n_cells_baseline'].sum()}")
print(f"    Cells after filtering: {stats_df['n_cells_filtered'].sum()}")
print(f"    Mean clonotype size (baseline): {stats_df['n_cells_baseline'].mean():.1f}")
print(f"    Median clonotype size (baseline): {stats_df['n_cells_baseline'].median():.1f}")

print(f"\n  LY49C DISTRIBUTION:")
print(f"    Mean % Ly49C+ cells per clonotype: {stats_df['pct_ly49c_positive'].mean():.1f}%")
print(f"    Median % Ly49C+ cells per clonotype: {stats_df['pct_ly49c_positive'].median():.1f}%")
print(f"    Clonotypes with 0% Ly49C+ cells: {(stats_df['pct_ly49c_positive'] == 0).sum()}")
print(f"    Clonotypes with <10% Ly49C+ cells: {(stats_df['pct_ly49c_positive'] < 10).sum()}")
print(f"    Clonotypes with <25% Ly49C+ cells: {(stats_df['pct_ly49c_positive'] < 25).sum()}")
print(f"    Clonotypes with >50% Ly49C+ cells: {(stats_df['pct_ly49c_positive'] > 50).sum()}")

print(f"\n  ENTROPY - Specificity:")
print(f"    Mean entropy (baseline): {stats_df['entropy_baseline'].mean():.3f}")
print(f"    Mean entropy (filtered): {stats_df['entropy_filtered'].mean():.3f}")
print(f"    Mean entropy change: {stats_df['entropy_change'].mean():.4f}")

print(f"\n  DOMINANCE - Top peptide fraction:")
print(f"    Mean dominance (baseline): {stats_df['dominance_baseline'].mean():.3f}")
print(f"    Mean dominance (filtered): {stats_df['dominance_filtered'].mean():.3f}")
print(f"    Mean dominance change: {stats_df['dominance_change'].mean():.4f}")

print(f"\n  WITHIN-CLONOTYPE SIMILARITY:")
print(f"    Mean similarity (baseline): {stats_df['cosine_similarity_baseline'].mean():.4f}")
print(f"    Mean similarity (filtered): {stats_df['cosine_similarity_filtered'].mean():.4f}")
print(f"    Mean similarity change: {stats_df['similarity_change'].mean():.4f}")

print(f"\n  PATTERN DISTANCE:")
print(f"    Mean cosine distance: {stats_df['pattern_distance_cosine'].mean():.4f}")
print(f"    Median cosine distance: {stats_df['pattern_distance_cosine'].median():.4f}")
print(f"    Max cosine distance: {stats_df['pattern_distance_cosine'].max():.4f}")

print(f"\n  TOP PEPTIDES:")
top_peptide_counts = stats_df['top_peptide'].value_counts()
print(f"    Distribution of dominant peptides:")
for pep, count in top_peptide_counts.items():
    print(f"      {pep}: {count} clonotypes ({100*count/len(stats_df):.1f}%)")


  CLONOTYPE COUNTS:
    Total clonotypes (≥5 cells): 470
    Clonotypes retained after filtering: 465

  CELL COUNTS:
    Total cells in clonotypes: 12547
    Cells after filtering: 12267
    Mean clonotype size (baseline): 26.7
    Median clonotype size (baseline): 11.5

  LY49C DISTRIBUTION:
    Mean % Ly49C+ cells per clonotype: 26.4%
    Median % Ly49C+ cells per clonotype: 25.0%
    Clonotypes with 0% Ly49C+ cells: 37
    Clonotypes with <10% Ly49C+ cells: 42
    Clonotypes with <25% Ly49C+ cells: 219
    Clonotypes with >50% Ly49C+ cells: 18

  ENTROPY - Specificity:
    Mean entropy (baseline): 1.057
    Mean entropy (filtered): 1.055
    Mean entropy change: -0.0038

  DOMINANCE - Top peptide fraction:
    Mean dominance (baseline): 0.740
    Mean dominance (filtered): 0.740
    Mean dominance change: 0.0005

  WITHIN-CLONOTYPE SIMILARITY:
    Mean similarity (baseline): 0.8380
    Mean similarity (filtered): 0.8367
    Mean similarity change: -0.0003

  PATTERN DISTANCE:
    

In [50]:
common_cts = set(baseline_summaries.keys()) & set(final_summaries.keys())
sorted_common = sorted(common_cts, key=lambda x: baseline_summaries[x].n_cells, reverse=True)[:20]

xs = np.arange(len(peptides))
width = 0.35

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
axes = axes.flatten()

for ax_i, ct in enumerate(sorted_common):
    ax = axes[ax_i]
    b = baseline_summaries[ct]
    f = final_summaries[ct]

    ax.bar(xs - width/2, b.mean_pattern, width, label='Baseline', alpha=0.7, color='steelblue')
    ax.bar(xs + width/2, f.mean_pattern, width, label='Filtered', alpha=0.7, color='darkorange')

    dist = pattern_distance(b.mean_pattern, f.mean_pattern, 'cosine')

    ax.set_title(
        f"{ct[:30]}{'...' if len(ct) > 30 else ''}\n"
        f"n: {b.n_cells}→{f.n_cells}, Δ={dist:.3f}",
        fontsize=6
    )
    ax.set_xticks(xs)
    ax.set_xticklabels(peptides, rotation=90, fontsize=5)
    ax.tick_params(axis='y', labelsize=5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if ax_i == 0:
        ax.legend(fontsize=6)

for j in range(len(sorted_common), len(axes)):
    axes[j].axis('off')

fig.tight_layout()
plt.savefig(output_dir / 'before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved: {output_dir / 'before_after_comparison.png'}")

  Saved: Comparison3_Figures/before_after_comparison.png


In [51]:
def plot_ly49c_cumulative_by_clonotype(df, ct_col, ly49c_col, clonotypes, output_path):
    """
    For each clonotype, plot cumulative Ly49C distribution.
    """
    n_plots = len(clonotypes)
    ncols = 4
    nrows = (n_plots + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = axes.flatten()

    for ax_i, ct in enumerate(clonotypes):
        ax = axes[ax_i]

        ct_data = df[df[ct_col] == ct][ly49c_col].to_numpy(dtype=float, copy=False)
        ct_data_sorted = np.sort(ct_data)

        n_cells = len(ct_data_sorted)
        cumsum = np.cumsum(ct_data_sorted)
        x = np.arange(1, n_cells + 1)

        ax.plot(x, cumsum, 'b-', lw=2)
        ax.fill_between(x, 0, cumsum, alpha=0.3)

        n_positive = int(np.sum(ct_data_sorted > 0))
        pct_positive = 100 * n_positive / n_cells if n_cells else 0.0

        first_positive_idx = np.searchsorted(ct_data_sorted, 1, side='left')
        if first_positive_idx < n_cells:
            ax.axvline(first_positive_idx + 1, color='red', ls='--', alpha=0.7)

        ax.set_xlabel('Cells (sorted by Ly49C)', fontsize=8)
        ax.set_ylabel('Cumulative Ly49C', fontsize=8)
        ax.set_title(
            f"{ct[:35]}{'...' if len(ct) > 35 else ''}\n"
            f"n={n_cells}, {n_positive} cells Ly49C+ ({pct_positive:.1f}%)",
            fontsize=7
        )
        ax.tick_params(labelsize=6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    for j in range(len(clonotypes), len(axes)):
        axes[j].axis('off')

    fig.suptitle('Ly49C Distribution Within Clonotypes\n(red line = first Ly49C+ cell)',
                 fontsize=12, y=1.02)
    fig.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()


top_clonotypes = sorted(
    baseline_summaries.keys(),
    key=lambda x: baseline_summaries[x].n_cells,
    reverse=True
)[:20]

plot_ly49c_cumulative_by_clonotype(
    df, ct_col, ly49c_col, top_clonotypes,
    output_dir / 'ly49c_cumulative_by_clonotype.png'
)
print(f"  Saved: {output_dir / 'ly49c_cumulative_by_clonotype.png'}")

  Saved: Comparison3_Figures/ly49c_cumulative_by_clonotype.png


In [52]:
ly49c_stats = []
for ct in baseline_summaries.keys():
    ct_ly49c = df_by_ct[ct][ly49c_col].to_numpy(dtype=float, copy=False)
    n_cells = len(ct_ly49c)
    n_positive = int(np.sum(ct_ly49c > 0))

    ly49c_stats.append({
        'clonotype': ct,
        'n_cells': n_cells,
        'n_ly49c_positive': n_positive,
        'pct_ly49c_positive': 100 * n_positive / n_cells if n_cells else 0.0,
        'total_ly49c': float(np.sum(ct_ly49c)),
        'mean_ly49c': float(np.mean(ct_ly49c)) if n_cells else 0.0,
        'max_ly49c': float(np.max(ct_ly49c)) if n_cells else 0.0
    })

ly49c_stats_df = pd.DataFrame(ly49c_stats)

print(f"\nAcross {len(ly49c_stats_df)} clonotypes:")
print(f"  Mean % cells with Ly49C > 0: {ly49c_stats_df['pct_ly49c_positive'].mean():.1f}%")
print(f"  Median % cells with Ly49C > 0: {ly49c_stats_df['pct_ly49c_positive'].median():.1f}%")
print(f"  Clonotypes with <10% Ly49C+ cells: {(ly49c_stats_df['pct_ly49c_positive'] < 10).sum()}")
print(f"  Clonotypes with <25% Ly49C+ cells: {(ly49c_stats_df['pct_ly49c_positive'] < 25).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(ly49c_stats_df['pct_ly49c_positive'], bins=20, edgecolor='black', alpha=0.7)
ax.axvline(ly49c_stats_df['pct_ly49c_positive'].median(), color='red', ls='--',
           label=f"Median: {ly49c_stats_df['pct_ly49c_positive'].median():.1f}%")
ax.set_xlabel('% Cells with Ly49C > 0')
ax.set_ylabel('Number of Clonotypes')
ax.set_title('Distribution of Ly49C+ Cell Fraction')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
ax.scatter(ly49c_stats_df['n_cells'], ly49c_stats_df['pct_ly49c_positive'], alpha=0.5, s=20)
ax.set_xlabel('Clonotype Size (n cells)')
ax.set_ylabel('% Cells with Ly49C > 0')
ax.set_title('Ly49C+ Fraction vs Clonotype Size')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_dir / 'ly49c_fraction_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved: {output_dir / 'ly49c_fraction_summary.png'}")

ly49c_stats_df.to_csv(output_dir / 'ly49c_clonotype_stats.csv', index=False)
print(f"  Saved: {output_dir / 'ly49c_clonotype_stats.csv'}")


Across 470 clonotypes:
  Mean % cells with Ly49C > 0: 26.4%
  Median % cells with Ly49C > 0: 25.0%
  Clonotypes with <10% Ly49C+ cells: 42
  Clonotypes with <25% Ly49C+ cells: 219
  Saved: Comparison3_Figures/ly49c_fraction_summary.png
  Saved: Comparison3_Figures/ly49c_clonotype_stats.csv


In [53]:
common_cts = set(baseline_summaries.keys()) & set(final_summaries.keys())

consistency_data = []
for ct in common_cts:
    consistency_data.append({
        'clonotype': ct,
        'similarity_baseline': baseline_summaries[ct].cosine_similarity,
        'similarity_filtered': final_summaries[ct].cosine_similarity,
        'n_baseline': baseline_summaries[ct].n_cells,
        'n_filtered': final_summaries[ct].n_cells
    })

consistency_df = pd.DataFrame(consistency_data).dropna()
consistency_df['similarity_change'] = consistency_df['similarity_filtered'] - consistency_df['similarity_baseline']

print(f"Mean within-clonotype similarity (baseline): {consistency_df['similarity_baseline'].mean():.4f}")
print(f"Mean within-clonotype similarity (filtered): {consistency_df['similarity_filtered'].mean():.4f}")
print(f"Mean change: {consistency_df['similarity_change'].mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(consistency_df['similarity_baseline'], consistency_df['similarity_filtered'], alpha=0.5, s=20)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='No change')
ax.set_xlabel('Within-Clonotype Cosine Similarity (Baseline)')
ax.set_ylabel('Within-Clonotype Cosine Similarity (Filtered)')
ax.set_title('Effect of Ly49C Filtering on Clonotype Coherence')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / 'consistency_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved: {output_dir / 'consistency_comparison.png'}")

Mean within-clonotype similarity (baseline): 0.8369
Mean within-clonotype similarity (filtered): 0.8367
Mean change: -0.0003
  Saved: Comparison3_Figures/consistency_comparison.png


In [54]:
print(f"\nThreshold selected: q = {CHOSEN_PERCENTILE:.3f}")

print("\nPer-sample thresholds:")
for sample in df[sample_col].unique():
    n_before = len(df[df[sample_col] == sample])
    n_after = len(final_df[final_df[sample_col] == sample])
    threshold = df[df[sample_col] == sample][ly49c_col].quantile(CHOSEN_PERCENTILE)
    print(f"  {sample}: T_s = {threshold:.1f}, {n_before}→{n_after} cells ({100*n_after/n_before:.1f}%)")

print("\nClonotype-level summary:")
print(f"  Baseline: {len(baseline_summaries)} clonotypes")
print(f"  Filtered: {len(final_summaries)} clonotypes")

sweep_df.to_csv(output_dir / 'threshold_sweep_results.csv', index=False)
consistency_df.to_csv(output_dir / 'consistency_analysis.csv', index=False)

print(f"\n✓ All outputs saved to: {output_dir}/")


Threshold selected: q = 0.950

Per-sample thresholds:
  BL6-B10BR_HTxC: T_s = 2.0, 10852→10629 cells (97.9%)
  BL6-B10BR_HTxB: T_s = 2.0, 7501→7289 cells (97.2%)
  BL6-B10BR_HTxA: T_s = 2.0, 571→550 cells (96.3%)

Clonotype-level summary:
  Baseline: 470 clonotypes
  Filtered: 465 clonotypes

✓ All outputs saved to: Comparison3_Figures/
